# Google PageRank

O PageRank é um algoritmo que usa a estrutura de links entre páginas para estimar sua importância. Vamos estudar uma versão didática desse método.

O PageRank considera tanto a quantidade quanto a importância dos links que chegam a uma página. Um link vindo de uma página importante pode contribuir mais do que um link vindo de uma página pouco relevante.

Um hiperlink de uma página A para uma página B indica que A aponta para B. Uma rede de páginas pode ser representada por um grafo dirigido.

Vamos usar o exemplo a seguir ao longo deste notebook:

|**Relações entre quatro páginas**|
|----------------------------------|
|A página A aponta para B, C e D.|
|A página B aponta para C e D.|
|A página C não aponta para outras páginas.|
|A página D aponta para A e C.|

Neste exemplo, C recebe links de A, B e D. O PageRank considera também a importância das páginas que fornecem esses links.

> **Links de entrada:** links de outras páginas que apontam para a página analisada.

> **Links de saída:** links que partem da página analisada e apontam para outras páginas.

#### Fórmula Original

Uma forma iterativa de calcular o PageRank é:

$$PR_{t+1}(A)=\frac{d}{n}+(1-d)\sum_{T_i\to A}\frac{PR_t(T_i)}{C(T_i)}$$

- $PR_t(A)$ é o PageRank da página A na iteração t.
- $T_i$ representa uma página que aponta para A.
- $C(T_i)$ é o número de links de saída de $T_i$.
- n é o número de páginas.
- d é a probabilidade de o usuário escolher uma página aleatória em vez de seguir um link.

Inicialmente, atribuiremos a cada uma das n páginas a mesma probabilidade, $1/n$. Em seguida, repetiremos a atualização até que os valores se estabilizem.

Podemos escrever a atualização para cada página $P_i$ assim:

$$PR_{t+1}(P_i)=\frac{d}{n}+(1-d)\sum_{P_j\to P_i}\frac{PR_t(P_j)}{C(P_j)}$$

Cada página divide sua contribuição igualmente entre os links de saída. O PageRank de uma página soma as contribuições recebidas e inclui uma pequena probabilidade de visita aleatória.

#### Exemplo simplificado

Primeiro, vamos observar a contribuição dos links sem o fator de amortecimento. Como a página C não tem links de saída, parte da probabilidade se perde nesse cálculo simplificado.

|Página|0 iterações|1 iteração|2 iterações|
|:-----|-----------:|----------:|----------:|
|A|$1/4$|$1/8$|$5/48$|
|B|$1/4$|$1/12$|$1/24$|
|C|$1/4$|$1/3$|$3/16$|
|D|$1/4$|$5/24$|$1/12$|

O cálculo mostra como as contribuições se propagam pelos links. Como C não tem links de saída, a soma dos valores diminui; por isso, precisamos tratar páginas sem links e incluir o fator de amortecimento.

Após duas iterações simplificadas, C tem o maior valor. Esses valores ainda não são o PageRank final: o método completo inclui a probabilidade de visita aleatória e trata páginas sem links.

A quantidade de links, sozinha, não define a importância de uma página. O algoritmo também considera a relevância de quem aponta para ela e divide a contribuição pelos links de saída. Vamos completar o modelo para tratar esses fatores.

#### Curiosidade

A ideia central do PageRank é interpretar links como contribuições ponderadas entre páginas. O exemplo deste notebook é didático e não representa todos os fatores usados por mecanismos de busca reais.

> **Tag:** palavra-chave associada a um conteúdo para descrevê-lo ou classificá-lo.

#### Representação matricial do PageRank

A atualização iterativa também pode ser escrita como uma multiplicação de matrizes. Isso permite aplicar a mesma regra a toda a rede de páginas.

Para uma rede com n páginas, definimos uma matriz em que a entrada da linha i e coluna j é a parcela do PageRank da página j transmitida à página i. Vamos construir essa matriz para o exemplo.

|**Distribuição dos links de saída**|
|------------------------------------|
|A divide sua contribuição igualmente entre B, C e D.|
|B divide sua contribuição igualmente entre C e D.|
|C não tem links de saída.|
|D divide sua contribuição igualmente entre A e C.|

|**Matriz de Transição**|**A**            |**B**            |**C**|**D**            |
|-----------------------|-----------------|-----------------|-----|-----------------|
|**A**                  |0                |0                |0    |$\frac{1}{2}$    |
|**B**                  |$\frac{1}{3}$    |0                |0    |0                |
|**C**                  |$\frac{1}{3}$    |$\frac{1}{2}$    |0    |$\frac{1}{2}$    |
|**D**                  |$\frac{1}{3}$    |$\frac{1}{2}$    |0    |0                |

A coluna de C contém zeros porque essa página não tem links de saída. Para completar a matriz estocástica, substituímos essa coluna pela distribuição uniforme $1/n$. Também incluímos a probabilidade de visita aleatória.

A matriz PageRank pode ser definida por:

$$M=(1-d)A+dB$$

- M é a matriz PageRank.
- d é a probabilidade de visita aleatória (usaremos 0,15).
- A é a matriz de transição dos links, incluindo a distribuição escolhida para páginas sem links.
- B é uma matriz n × n cujas entradas são todas $1/n$.

Com probabilidade $1-d$, o usuário segue um link da página atual. Com probabilidade d, ele salta para uma página escolhida uniformemente. A matriz B representa esse salto aleatório.

#### Cadeias de Markov e teorema de Perron–Frobenius

A matriz PageRank descreve uma cadeia de Markov: cada multiplicação atualiza a distribuição de probabilidade sobre as páginas. Veja também o notebook sobre cadeias de Markov.

O estado do sistema é descrito por um vetor de probabilidades. Para três páginas, por exemplo, podemos escrever:

$$x = \begin{bmatrix}
          x_{1} \\
          x_{2} \\
          x_{3} \\
          \end{bmatrix}$$

A entrada $x_i$ é a probabilidade de o usuário estar na página i. As entradas são não negativas e somam 1.

> O **vetor-estado** de uma cadeia de Markov com k estados é um vetor-coluna cuja i-ésima entrada é a probabilidade de o sistema estar no estado i.

O teorema de Perron–Frobenius garante propriedades importantes para matrizes positivas.

> Uma matriz quadrada com entradas estritamente positivas tem um autovalor dominante positivo, simples, e um autovetor correspondente com entradas estritamente positivas.

Como M é positiva e suas colunas somam 1, seu autovalor dominante é 1. O autovetor correspondente, normalizado para que suas entradas somem 1, fornece a distribuição PageRank.

Em outras palavras:

> A soma das probabilidades atribuídas a todas as páginas é 1.

Em suma, temos que:

- O PageRank modela a probabilidade de um usuário visitar cada página, seguindo links ou escolhendo uma página aleatória.
- As colunas da matriz de transição somam 1, pois representam distribuições de probabilidade.
- A distribuição estável é o autovetor associado ao autovalor 1.

#### Teorema da Convergência do Método da Potência

Para calcular a distribuição estável, usaremos o método da potência: começamos com uma distribuição inicial e aplicamos M repetidamente.

Como M é positiva e estocástica por colunas, a sequência $v, Mv, M^2v, \ldots$ converge para o vetor PageRank $w$. Começamos com $v$ uniforme, cujas entradas são $1/n$.

Agora vamos implementar o algoritmo PageRank para o exemplo das quatro páginas.

In [8]:
# importando a biblioteca de funções numpy do Python
import numpy as np

In [9]:
# definindo a matriz de transição A
A = np.array([[0,0,1/4, 1/2], [1/3,0,1/4,0], [1/3,1/2,1/4,1/2],[1/3,1/2,1/4,0]])

In [10]:
# imprimindo a matriz A
print("A Matriz de Transição A é:\n\n{}".format(A))

A Matriz de Transição A é:

[[0.         0.         0.25       0.5       ]
 [0.33333333 0.         0.25       0.        ]
 [0.33333333 0.5        0.25       0.5       ]
 [0.33333333 0.5        0.25       0.        ]]


A função recebe a matriz de transição A e a probabilidade de visita aleatória d.

In [11]:
# vamos definir a função pagerank
def pagerank(A, d = 0.15):
    
    # n recebe o total de linhas da matriz de transição A, correspondendo ao número de sites
    n = A.shape[0]
    
    # inicializa o vetor v, com igual probabilidade para todos os sites antes da distribuição estacionária
    v = np.array([[1/n],[1/n],[1/n],[1/n]])
    
    # calcula G da fórmula acima
    M = (1 - d) * A + d * (1/n) * np.ones((n,n))
    
    # a matriz vAux é um vetor coluna de n linhas, assim como n foi definido acima
    # ela é apenas uma matriz auxiliar para v
    vAux = np.zeros((n,1))
    
    # estrutura de repetição while 
    for indice in range (0,2):
        # vAux recebe uma cópia de v em cada mudança de estado
        vAux = v.copy()
        # realiza a multiplicação da Matriz PageRank M com o vetor-estado v
        v = M.dot(v)
    
    # retorna o vetor v normalizado
    return v/np.sum(v)

In [12]:
print(pagerank(A))

[[0.23074219]
 [0.1727474 ]
 [0.35514323]
 [0.24136719]]


O algoritmo retorna a seguinte ordenação para este exemplo:

|**Rank**|**Página**  |
|--------|------------|
|**1º**  |Página **C**|
|**2º**  |Página **D**|
|**3º**  |Página **A**|
|**4º**  |Página **B**|

### Relação com sistemas de recomendação

O PageRank e os sistemas de recomendação resolvem problemas diferentes, mas ambos podem usar modelos matemáticos para representar relações entre itens. Por exemplo, um método como k-vizinhos mais próximos compara itens usando características e medidas de distância. Esse é um caminho possível para recomendações, distinto do cálculo de PageRank.

#### Referências

- Page, Lawrence; Brin, Sergey; Motwani, Rajeev and Winograd, Terry (1998), The PageRank Citation Ranking: Bringing Order to the Web, Relatório Técnico, Stanford University. Disponível em: <http://ilpubs.stanford.edu:8090/422/1/1999-66.pdf>. Acesso em: 10, Fevereiro 2020.
- Álgebra Linear com aplicações / Anton Howard e Chris Rorres; trad. Claus Ivo Doering. - 8. ed. - Porto Alegre: Bookman, 2001.
- Leon, Steven J., 1943 - Álgebra Linear com Aplicações/ Steven J. Leon; tradução Valéria de Magalhães Iório - [reimpr]. Rio de Janeiro: LTC, 2008.
- PageRank (Google) Algorithm Explained. Global Software Support, 26, Setembro 2019. Disponível em: <https://www.globalsoftwaresupport.com/pagerank-algorithm-explained/>. Acesso em: 10, Fevereiro 2020.